In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from prophet import Prophet

In [ ]:
df = pd.read_csv("/content/covid_19_clean_complete.csv")

In [ ]:
df

,Province/State,Country/Region,Lat,Long,Date,Confirmed,Deaths,Recovered,Active,WHO Region
0,NaN,Afghanistan,33.939110,67.709953,2020-01-22,0,0,0,0,Eastern Mediterranean
1,NaN,Albania,41.153300,20.168300,2020-01-22,0,0,0,0,Europe
2,NaN,Algeria,28.033900,1.659600,2020-01-22,0,0,0,0,Africa
3,NaN,Andorra,42.506300,1.521800,2020-01-22,0,0,0,0,Europe
4,NaN,Angola,-11.202700,17.873900,2020-01-22,0,0,0,0,Africa
...,...,...,...,...,...,...,...,...,...,...
49063,NaN,Sao Tome and Principe,0.186400,6.613100,2020-07-27,865,14,734,117,Africa
49064,NaN,Yemen,15.552727,48.516388,2020-07-27,1691,483,833,375,Eastern Mediterranean
49065,NaN,Comoros,-11.645500,43.333300,2020-07-27,354,7,328,19,Africa
49066,NaN,Tajikistan,38.861000,71.276100,2020-07-27,7235,60,6028,1147,Europe


In [ ]:
df.shape

(49068, 10)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49068 entries, 0 to 49067
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Province/State  14664 non-null  object 
 1   Country/Region  49068 non-null  object 
 2   Lat             49068 non-null  float64
 3   Long            49068 non-null  float64
 4   Date            49068 non-null  object 
 5   Confirmed       49068 non-null  int64  
 6   Deaths          49068 non-null  int64  
 7   Recovered       49068 non-null  int64  
 8   Active          49068 non-null  int64  
 9   WHO Region      49068 non-null  object 
dtypes: float64(2), int64(4), object(4)
memory usage: 3.7+ MB


In [ ]:
df.isnull().sum()

,0
Province/State,34404
Country/Region,0
Lat,0
Long,0
Date,0
Confirmed,0
Deaths,0
Recovered,0
Active,0
WHO Region,0


In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49068 entries, 0 to 49067
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Province/State  14664 non-null  object        
 1   Country/Region  49068 non-null  object        
 2   Lat             49068 non-null  float64       
 3   Long            49068 non-null  float64       
 4   Date            49068 non-null  datetime64[ns]
 5   Confirmed       49068 non-null  int64         
 6   Deaths          49068 non-null  int64         
 7   Recovered       49068 non-null  int64         
 8   Active          49068 non-null  int64         
 9   WHO Region      49068 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(4), object(3)
memory usage: 3.7+ MB


In [ ]:
daily_cases = df.groupby('Date').agg({'Confirmed': 'sum', 'Deaths': 'sum', 'Recovered': 'sum'}).reset_index()
daily_cases

,Date,Confirmed,Deaths,Recovered
0,2020-01-22,555,17,28
1,2020-01-23,654,18,30
2,2020-01-24,941,26,36
3,2020-01-25,1434,42,39
4,2020-01-26,2118,56,52
...,...,...,...,...
183,2020-07-23,15510481,633506,8710969
184,2020-07-24,15791645,639650,8939705
185,2020-07-25,16047190,644517,9158743
186,2020-07-26,16251796,648621,9293464


In [ ]:
daily_cases['Active'] = daily_cases['Confirmed'] - daily_cases['Deaths'] - daily_cases['Recovered']
daily_cases

,Date,Confirmed,Deaths,Recovered,Active
0,2020-01-22,555,17,28,510
1,2020-01-23,654,18,30,606
2,2020-01-24,941,26,36,879
3,2020-01-25,1434,42,39,1353
4,2020-01-26,2118,56,52,2010
...,...,...,...,...,...
183,2020-07-23,15510481,633506,8710969,6166006
184,2020-07-24,15791645,639650,8939705,6212290
185,2020-07-25,16047190,644517,9158743,6243930
186,2020-07-26,16251796,648621,9293464,6309711


In [ ]:
daily_cases['New Cases'] = daily_cases['Confirmed'].diff().fillna(0)
daily_cases

,Date,Confirmed,Deaths,Recovered,Active,New Cases
0,2020-01-22,555,17,28,510,0.0
1,2020-01-23,654,18,30,606,99.0
2,2020-01-24,941,26,36,879,287.0
3,2020-01-25,1434,42,39,1353,493.0
4,2020-01-26,2118,56,52,2010,684.0
...,...,...,...,...,...,...
183,2020-07-23,15510481,633506,8710969,6166006,282756.0
184,2020-07-24,15791645,639650,8939705,6212290,281164.0
185,2020-07-25,16047190,644517,9158743,6243930,255545.0
186,2020-07-26,16251796,648621,9293464,6309711,204606.0


In [ ]:
daily_cases['Recovery Rate'] = (daily_cases['Recovered'] / daily_cases['Confirmed']) * 100
daily_cases

,Date,Confirmed,Deaths,Recovered,Active,New Cases,Recovery Rate
0,2020-01-22,555,17,28,510,0.0,5.045045
1,2020-01-23,654,18,30,606,99.0,4.587156
2,2020-01-24,941,26,36,879,287.0,3.825717
3,2020-01-25,1434,42,39,1353,493.0,2.719665
4,2020-01-26,2118,56,52,2010,684.0,2.455146
...,...,...,...,...,...,...,...
183,2020-07-23,15510481,633506,8710969,6166006,282756.0,56.161824
184,2020-07-24,15791645,639650,8939705,6212290,281164.0,56.610347
185,2020-07-25,16047190,644517,9158743,6243930,255545.0,57.073812
186,2020-07-26,16251796,648621,9293464,6309711,204606.0,57.184228


In [ ]:
daily_cases['Death Rate'] = (daily_cases['Deaths'] / daily_cases['Confirmed']) * 100
daily_cases

,Date,Confirmed,Deaths,Recovered,Active,New Cases,Recovery Rate,Death Rate
0,2020-01-22,555,17,28,510,0.0,5.045045,3.063063
1,2020-01-23,654,18,30,606,99.0,4.587156,2.752294
2,2020-01-24,941,26,36,879,287.0,3.825717,2.763018
3,2020-01-25,1434,42,39,1353,493.0,2.719665,2.928870
4,2020-01-26,2118,56,52,2010,684.0,2.455146,2.644004
...,...,...,...,...,...,...,...,...
183,2020-07-23,15510481,633506,8710969,6166006,282756.0,56.161824,4.084374
184,2020-07-24,15791645,639650,8939705,6212290,281164.0,56.610347,4.050560
185,2020-07-25,16047190,644517,9158743,6243930,255545.0,57.073812,4.016385
186,2020-07-26,16251796,648621,9293464,6309711,204606.0,57.184228,3.991073


**Visualization with plotly Library**

In [ ]:
# Total cases trend

fig1 = px.line(daily_cases,
    x='Date',
    y=['Confirmed', 'Recovered', 'Deaths', 'Active'],
    title='COVID-19 Global Trend Analysis')
fig1.show()

In [ ]:
# New infection trend

fig2 = px.line(daily_cases,
    x='Date',
    y='New Cases',
    title='Daily New Infection Trend')
fig2.show()

In [ ]:
# Recovery rate trend
fig3 = px.line(daily_cases,
    x='Date',
    y='Recovery Rate',
    title='Rate of Recovery Trend (%)')
fig3.show()

**Visualization with Prophet Library**

In [ ]:
prophet_data = daily_cases[['Date', 'Confirmed']]
prophet_data.columns = ['ds', 'y']

In [ ]:
prophet_data

,ds,y
0,2020-01-22,555
1,2020-01-23,654
2,2020-01-24,941
3,2020-01-25,1434
4,2020-01-26,2118
...,...,...
183,2020-07-23,15510481
184,2020-07-24,15791645
185,2020-07-25,16047190
186,2020-07-26,16251796


In [ ]:
model = Prophet()
model.fit(prophet_data)

INFO:prophet:Disabling yearly seasonality. Run prophet with yearly_seasonality=True to override this.
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


In [ ]:
future = model.make_future_dataframe(periods=7)
forecast = model.predict(future)

In [ ]:
print(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(7))

            ds          yhat    yhat_lower    yhat_upper
188 2020-07-28  1.632021e+07  1.620467e+07  1.642611e+07
189 2020-07-29  1.652998e+07  1.642585e+07  1.663847e+07
190 2020-07-30  1.674392e+07  1.663150e+07  1.685155e+07
191 2020-07-31  1.695911e+07  1.685086e+07  1.706387e+07
192 2020-08-01  1.716677e+07  1.706987e+07  1.727841e+07
193 2020-08-02  1.736430e+07  1.724979e+07  1.748115e+07
194 2020-08-03  1.755889e+07  1.743997e+07  1.767608e+07


In [ ]:
fig4 = go.Figure()

In [ ]:
# Actual data
fig4.add_trace(go.Scatter(
    x=prophet_data['ds'],
    y=prophet_data['y'],
    mode='lines',
    name='Actual Confirmed Cases'))

In [ ]:
# Predicted data
fig4.add_trace(go.Scatter(
    x=forecast['ds'],
    y=forecast['yhat'],
    mode='lines',
    name='Predicted Cases'))

In [ ]:
# Confidence interval
fig4.add_trace(go.Scatter(
    x=forecast['ds'],
    y=forecast['yhat_upper'],
    mode='lines',
    name='Upper Bound',
    line=dict(dash='dot')))

In [ ]:
fig4.add_trace(go.Scatter(
    x=forecast['ds'],
    y=forecast['yhat_lower'],
    mode='lines',
    name='Lower Bound',
    line=dict(dash='dot')))

In [ ]:
fig4.update_layout(
    title='COVID-19 Forecast for Next 7 Days',
    xaxis_title='Date',
    yaxis_title='Confirmed Cases')

fig4.show()